# 1. Setup

In [ ]:
!pip install langchain transformers torch langchain-text-splitters -q

In [ ]:
import polars as pl

import os
from pathlib import Path

import polars as pl
from kaggle_secrets import UserSecretsClient

# Lấy token từ Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Thiết lập biến môi trường
os.environ["HF_TOKEN"] = hf_token

# Thay 'vietnamese-legal-documents' bằng tên tập dữ liệu chính xác của bạn trên Kaggle
dataset_path = '/kaggle/input/datasets/brdh7940/vietnamese-legal-documents' 

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# 2. EDA

In [ ]:
DATA_PATH = Path("/kaggle/input/datasets/brdh7940/vietnamese-legal-documents/merged/")
content     = pl.read_parquet(DATA_PATH / "merged_content.parquet")
metadata    = pl.read_parquet(DATA_PATH / "merged_metadata.parquet")
relationships = pl.read_parquet(DATA_PATH / "merged_relationships.parquet")

In [ ]:
dfs = {"content": content, "metadata": metadata, "relationships": relationships}

In [ ]:
for name, df in dfs.items():
    print(f"\n{'='*60}")
    print(f"[{name.upper()}]  shape={df.shape}  estimated_size={df.estimated_size('mb'):.2f} MB")
    print(f"{'='*60}")
    print("--- Schema ---")
    print(df.schema)
    print("\n--- Null counts ---")
    print(df.null_count())
    print("\n--- Sample (3 rows) ---")
    print(df.head(3))

In [ ]:
# ── Step 2: Value distributions ──────────────────────────────

# 2a. content: source distribution + content length stats
print("=== CONTENT: source value_counts ===")
print(content["source"].value_counts().sort("count", descending=True))

print("\n=== CONTENT: content length stats (chars) ===")
content_len = content.with_columns(pl.col("content").str.len_chars().alias("content_len"))
print(content_len["content_len"].describe())

# 2b. metadata: key categorical columns
cat_cols = ["legal_type", "effect_status", "source"]
for col in cat_cols:
    print(f"\n=== METADATA: {col} value_counts ===")
    print(metadata[col].value_counts().sort("count", descending=True).head(20))

# 2c. metadata: date range của issuance_date
print("\n=== METADATA: issuance_date sample (non-null) ===")
print(metadata.filter(pl.col("issuance_date").is_not_null())["issuance_date"].head(10))
print("\n=== METADATA: issuance_date - min/max ===")
dates = metadata.filter(pl.col("issuance_date").is_not_null())["issuance_date"]
print(f"Total non-null: {len(dates)}")
print(f"Sample unique formats: {dates.unique().head(5).to_list()}")

# 2d. relationships: relationship type distribution
print("\n=== RELATIONSHIPS: relationship value_counts ===")
print(relationships["relationship"].value_counts().sort("count", descending=True))

# 2e. overlap: bao nhiêu content.id có trong metadata.id?
content_ids  = set(content["id"].to_list())
metadata_ids = set(metadata["id"].to_list())
print(f"\n=== ID OVERLAP ===")
print(f"content ids:       {len(content_ids):,}")
print(f"metadata ids:      {len(metadata_ids):,}")
print(f"intersection:      {len(content_ids & metadata_ids):,}")
print(f"content only:      {len(content_ids - metadata_ids):,}")
print(f"metadata only:     {len(metadata_ids - content_ids):,}")

In [ ]:
# ── Step 3: Duplicates, content length buckets, date range ───

# 3a. Duplicate IDs trong content
print("=== CONTENT: duplicate id analysis ===")
dup_counts = content.group_by("id").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
print(f"IDs xuất hiện > 1 lần: {dup_counts.shape[0]:,}")
print(f"Tổng rows dư (duplicates): {(dup_counts['n'] - 1).sum():,}")
print("\nTop 5 IDs xuất hiện nhiều nhất:")
print(dup_counts.sort("n", descending=True).head(5))

print("\nSample duplicate rows (cùng id, khác source):")
if dup_counts.shape[0] > 0:
    sample_dup_id = dup_counts.sort("n", descending=True).head(1)["id"][0]
    print(content.filter(pl.col("id") == sample_dup_id).select(["id", "source", pl.col("content").str.slice(0, 100)]))
else:
    print("✅ Không có duplicate — đã được xử lý ở bước preprocess.")

# 3b. Content length buckets
print("\n=== CONTENT: length distribution buckets ===")
buckets = content.with_columns(pl.col("content").str.len_chars().alias("L")).with_columns(
    pl.when(pl.col("L") < 500).then(pl.lit("< 500"))
     .when(pl.col("L") < 2000).then(pl.lit("500–2K"))
     .when(pl.col("L") < 5000).then(pl.lit("2K–5K"))
     .when(pl.col("L") < 20000).then(pl.lit("5K–20K"))
     .when(pl.col("L") < 100000).then(pl.lit("20K–100K"))
     .otherwise(pl.lit("> 100K")).alias("bucket")
)
print(buckets["bucket"].value_counts().sort("count", descending=True))

# 3c. Metadata: issuance_date actual min/max (parse as date)
print("\n=== METADATA: issuance_date min/max (true sort) ===")
date_df = metadata.filter(pl.col("issuance_date").is_not_null()).with_columns(
    pl.col("issuance_date").str.to_date(format="%Y-%m-%d", strict=False).alias("date_parsed")
).filter(pl.col("date_parsed").is_not_null())
print(f"Parseable dates: {date_df.shape[0]:,}")
print(f"Min date: {date_df['date_parsed'].min()}")
print(f"Max date: {date_df['date_parsed'].max()}")

print("\n=== METADATA: documents per year (top 20) ===")
year_dist = date_df.with_columns(pl.col("date_parsed").dt.year().alias("year")) \
    .group_by("year").agg(pl.len().alias("count")) \
    .sort("year", descending=True).head(20)
print(year_dist)

# 3d. legal_sectors: có bao nhiêu unique sectors (non-null)
print("\n=== METADATA: legal_sectors non-null sample ===")
sectors_non_null = metadata.filter(pl.col("legal_sectors").is_not_null())
print(f"Non-null legal_sectors: {sectors_non_null.shape[0]:,}")
print("Sample values:")
print(sectors_non_null["legal_sectors"].head(10))

# 3e. issuing_authority top 20
print("\n=== METADATA: issuing_authority top 20 ===")
print(metadata.filter(pl.col("issuing_authority").is_not_null())
      ["issuing_authority"].value_counts().sort("count", descending=True).head(20))

In [ ]:
# ── Step 4: Final stats before generating markdown ────────────

# 4a. legal_sectors: tách multi-value, đếm unique sectors
print("=== METADATA: legal_sectors unique values (split by ',') ===")
sectors_exploded = (
    metadata.filter(pl.col("legal_sectors").is_not_null())
    .select(pl.col("legal_sectors").str.split(", ").alias("s"))
    .explode("s")
    .with_columns(pl.col("s").str.strip_chars())
)
sector_counts = sectors_exploded["s"].value_counts().sort("count", descending=True)
print(f"Total unique sectors: {sector_counts.shape[0]}")
print(sector_counts.head(25))

# 4b. Relationships: degree stats (số quan hệ mỗi document)
print("\n=== RELATIONSHIPS: out-degree per doc_id (top 10) ===")
out_degree = relationships.group_by("doc_id").agg(pl.len().alias("n_relations"))
print(out_degree["n_relations"].describe())
print("\nTop 10 docs có nhiều relations nhất:")
print(out_degree.sort("n_relations", descending=True).head(10))

# 4c. Relationships: số docs tham gia trong graph
all_nodes = set(relationships["doc_id"].to_list()) | set(relationships["other_doc_id"].to_list())
print(f"\nTotal unique doc nodes in graph: {len(all_nodes):,}")
print(f"Metadata docs có trong graph:   {len(all_nodes & metadata_ids):,}")

# 4d. Null summary table cho metadata (% null mỗi cột)
print("\n=== METADATA: % null per column ===")
total = metadata.shape[0]
null_pct = metadata.null_count().transpose(include_header=True, column_names=["null_count"])
null_pct = null_pct.with_columns(
    (pl.col("null_count") / total * 100).round(2).alias("null_pct")
).sort("null_count", descending=True)
print(null_pct)

# 4e. content: check xem duplicate rows là exact duplicate hay khác content
print("\n=== CONTENT: duplicate rows — exact content match? ===")
dup_ids = content.group_by("id").agg(pl.len().alias("n")).filter(pl.col("n") > 1)["id"]
dup_rows = content.filter(pl.col("id").is_in(dup_ids))
exact_dups = dup_rows.group_by(["id","content"]).agg(pl.len().alias("n")).filter(pl.col("n") > 1)
print(f"Duplicate IDs total:           {len(dup_ids):,}")
print(f"Trong đó exact content match:  {exact_dups.shape[0]:,}")
print(f"Khác content (cần xem xét):    {len(dup_ids) - exact_dups.shape[0]:,}")

# 3. Preprocess

In [ ]:
# === Preprocess ===

# Bước 1: Deduplicate content (593,409 → 564,136 rows)
content = content.unique(subset=["id"])

# Bước 2: Drop cột 100% null khỏi metadata
metadata = metadata.drop("thong_tin_ap_dung")

# Bước 3: Loại bỏ văn bản không còn hiệu lực (giữ lại null effect_status)
INACTIVE_STATUSES = ["Hết hiệu lực", "Ngưng hiệu lực", "Chưa có hiệu lực"]
metadata = metadata.filter(~pl.col("effect_status").is_in(INACTIVE_STATUSES))

# Bước 4: Sync content và relationships theo metadata ids còn lại
active_ids = set(metadata["id"].to_list())

content = content.filter(pl.col("id").is_in(active_ids))

relationships = relationships.filter(
    pl.col("doc_id").is_in(active_ids) & pl.col("other_doc_id").is_in(active_ids)
)

In [ ]:
CLEAN_PATH = Path("/kaggle/working/cleaned")
os.makedirs(CLEAN_PATH, exist_ok=True)

content.write_parquet(CLEAN_PATH / "content.parquet")
metadata.write_parquet(CLEAN_PATH / "metadata.parquet")
relationships.write_parquet(CLEAN_PATH / "relationships.parquet")

In [ ]:
import shutil

# Đường dẫn đến thư mục bạn muốn tải
folder_path = '/kaggle/working/cleaned'
# Tên file zip sau khi nén
output_filename = 'cleaned_data'

# Nén thư mục thành file .zip
shutil.make_archive(output_filename, 'zip', folder_path)

print(f"Đã nén xong thư mục vào: {output_filename}.zip")